In [7]:
import numpy as np
from scipy import constants
import pandas as pd
import matplotlib.pyplot as plt
from tweezer_functions import * 
from IonChainTools import *
from scipy.optimize import fsolve
import matplotlib.colors as mcolors
import matplotlib.colorbar as mcolorbar
from scipy.optimize import fsolve
from scipy.optimize import curve_fit
from scipy.optimize import minimize
import matplotlib.ticker as ticker


#Constants in SI units
eps0 = constants.epsilon_0 
m = 39.9626*constants.atomic_mass
c = constants.c
e = constants.e
hbar = constants.hbar
pi = np.pi

# setting up parameters that we're not changing
qubit_wavelength = 729e-9
tweezer_wavelength = 532e-9
omega_tweezer = 2*pi*c/tweezer_wavelength
print(omega_tweezer)
df = pd.read_csv("S_P_only.csv",sep = ",",encoding = "UTF-8")
lambdares = np.array(df["wavelength (nm)"])*1e-9
omega_res = 2*pi*c/lambdares
linewidths = np.array(df["A_ki (s^-1)"])
lifetimes = linewidths
print(linewidths)


3540698434791077.0
[1.47e+08 1.40e+08]


In [8]:
def tweezer_combos_full_radial_test(
    omega_tweezer,
    linewidths,
    omega_res,
    m,
    mode_calc_r,
    N_list,
    f_rf_r,
    f_rf_a,
    P_opt,
    w0,
    max_tweezed=1,
    qubit_lambda=729e-9,
):
    """
    As before, but compute k = 2*pi/qubit_lambda and scale eigenvectors by
    k * sqrt(hbar / (2 * m * omega_mode)), where omega_mode = 2*pi*freq_Hz.
    """

    if np.isscalar(N_list):
        N_list = [int(N_list)]
    if np.isscalar(P_opt):
        P_opt = [P_opt]

    pi = np.pi
    rows = []
    # compute k from provided qubit wavelength and base scale
    k = 2 * pi / qubit_lambda
    eigvec_scale = k * np.sqrt(hbar / (2 * m))

    for N in N_list:
        # Generate all possible tweezer combinations, but only over the first N/2 ions
        half_range = N // 2
        max_tweezed_local = min(max_tweezed, half_range)
        all_positions = range(half_range)
        all_combos = []
        for r in range(0, max_tweezed_local + 1):
            all_combos.extend(itertools.combinations(all_positions, r))

        w_rf_r = f_rf_r * 2 * pi
        w_rf_r_list = np.full(N, w_rf_r)
        ueq = ion_spacing(N, f_rf_a)[0]

        for P_total in P_opt:
            for tweezed_positions in all_combos:
                n_tweezed = len(tweezed_positions)
                P_per = P_total / n_tweezed if n_tweezed > 0 else 0.0

                pot = potential(omega_tweezer, linewidths, omega_res, P_per, w0)
                w_tw_r = omega_tweezer_r(pot, w0, m)

                combo = np.array([
                    np.sqrt(w_tw_r**2 + w_rf_r_list[i]**2) if i in tweezed_positions else w_rf_r_list[i]
                    for i in range(N)
                ])

                modes = mode_calc_r(m, combo, ueq, N)

                freqs = np.array([f for f, v in modes], dtype=float) if len(modes) else np.array([], dtype=float)
                if len(modes):
                    eigvecs = np.vstack([np.ravel(v) for f, v in modes])
                else:
                    eigvecs = np.empty((0, N))

                row = {
                    "N": N,
                    "Tweezed ions": tweezed_positions,
                    "P_per_tweezer (W)": P_per,
                    "Combined radial frequencies": combo,
                }

                for mode_index in range(N):
                    if mode_index < len(freqs):
                        freq_val = float(freqs[mode_index])  # freq in Hz as returned by mode_calc_r
                        row[f"Mode{mode_index}_freq"] = freq_val
                    else:
                        freq_val = np.nan
                        row[f"Mode{mode_index}_freq"] = np.nan

                    if mode_index < eigvecs.shape[0] and not np.isnan(freq_val) and freq_val > 0:
                        # convert to angular frequency (rad/s)
                        omega_mode = 2.0 * pi * freq_val
                        # final scale uses angular frequency (zero-point amplitude)
                        scale = eigvec_scale * np.sqrt(1.0 / omega_mode)
                        row[f"Mode{mode_index}_eigvec"] = np.ravel(eigvecs[mode_index]).astype(float) * scale
                    else:
                        row[f"Mode{mode_index}_eigvec"] = np.full(N, np.nan, dtype=float)

                rows.append(row)

    return pd.DataFrame(rows)

In [9]:
import numpy as np
import math

def run_optimal_mode_selection_tweezed_only_test(
    omega_tweezer,
    linewidths,
    omega_res,
    m,
    mode_calc_r,
    N,
    f_rf_r,
    f_rf_a,
    P_opt,
    w0,
    max_tweezed=1
):
    """
    Run optimal mode selection for all tweezed configurations only.
    
    Returns:
        winners: list of tuples
                 [((tweezed_ion, mapping, max_of_min_abs), (best_ion_index, best_mode_index)), ...]
    """
    
    # 1. Build Lamb-Dicke parameter lists for all configurations
    df = tweezer_combos_full_radial_test(
        omega_tweezer, linewidths, omega_res, m,
        mode_calc_r, N, f_rf_r, f_rf_a, P_opt, w0,
        max_tweezed=max_tweezed
    )
    
    result = build_mode_series_and_combinations(df)
    mode_lists_dict = result["mode_lists"]
    
    # ---- CLEAN: remove unusable keys ----
    cleaned = {
        k: v for k, v in mode_lists_dict.items()
        if k is not None and not (isinstance(k, float) and math.isnan(k)) and k != ()
    }
    mode_lists_dict = cleaned
    
    if not mode_lists_dict:
        return []
    
    # 2. Sort mode indices and collect all mode lists
    mode_indices = sorted(
        mode_lists_dict.keys(),
        key=lambda x: (str(x) if isinstance(x, tuple) else x)
    )
    mode_lists_ordered = [mode_lists_dict[i] for i in mode_indices]
    
    # 3. Combine modes across all tweezed ions
    combos = combine_lists(*mode_lists_ordered)
    
    # 4. Condense by min(abs)
    condensed = {k: condense_by_min_abs(v) for k, v in combos.items()}
    
    # 5. Filter by max(min(abs)) within each group
    best_each = {k: filter_by_max_min_abs(v) for k, v in condensed.items()}
    
    # 6. Select global winners
    winners = select_global_max_min_abs(list(best_each.values()))

    # 7. Compute best ion/mode indices properly
    augmented = []
    for tweezed_ion, mapping, score in winners:
        # Access the corresponding values from combos
        combo_dict = {combo_indices: arr for _, combo_indices, arr in combos[tweezed_ion]}
        values = combo_dict.get(mapping, np.array(mapping))  # Fallback to mapping
        
        # best ion index = position in vector responsible for min(abs) (i.e., score)
        best_ion_index = int(np.argmin(np.abs(values)))
        best_mode_index = mapping[best_ion_index]

        # ✅ ADD P_opt BACK INTO THE RETURNED STRUCTURE
        augmented.append(
            ((tweezed_ion, mapping, score, P_opt), (best_ion_index, best_mode_index))
        )

    return augmented


def run_optimal_mode_selection_untweezed_only_test(
    omega_tweezer,
    linewidths,
    omega_res,
    m,
    mode_calc_r,
    N,
    f_rf_r,
    f_rf_a,
    P_opt,
    w0
):
    """
    Run optimal mode selection for the untweezed configuration only.
    
    Returns:
        winners: list of tuples
                 [(None, ion_to_mode_mapping, max_of_min_abs), ...]
    """
    
    # 1. Build Lamb-Dicke parameter lists for the untweezed configuration
    df = tweezer_combos_full_radial_test(
        omega_tweezer, linewidths, omega_res, m,
        mode_calc_r, N, f_rf_r, f_rf_a, P_opt, w0,
        max_tweezed=0
    )
    
    result = build_mode_series_and_combinations(df)
    mode_lists_dict = result["mode_lists"]
    
    # ---- CLEAN: remove unusable keys ----
    cleaned = {
        k: v for k, v in mode_lists_dict.items()
        if k is not None and not (isinstance(k, float) and math.isnan(k)) and k != ()
    }
    mode_lists_dict = cleaned
    
    if not mode_lists_dict:
        return []
    
    # 2. Sort mode indices numerically if possible
    mode_indices = sorted(
        mode_lists_dict.keys(),
        key=lambda x: (str(x) if isinstance(x, tuple) else x)
    )
    
    if not mode_indices:
        return []
    
    # 3. Collect lists in canonical order
    mode_lists_ordered = [mode_lists_dict[i] for i in mode_indices]
    
    # 4. Combine modes
    combos = combine_lists(*mode_lists_ordered)
    
    # 5. Condense by min(abs)
    condensed = {k: condense_by_min_abs(v) for k, v in combos.items()}
    
    # 6. Filter by max(min(abs)) within each group
    best_each = {k: filter_by_max_min_abs(v) for k, v in condensed.items()}
    
    # 7. Select global winners
    winners = select_global_max_min_abs(list(best_each.values()))
    
    # 8. Add placeholder for "none tweezed"
    winners = [(None, combo, val) for _, combo, val in winners]

    return winners

In [10]:
def _extract_score_from_win(win):
    """Return numeric score from a winner entry or None if not found."""
    if not isinstance(win, (list, tuple)):
        return None
    # nested format: ((ion, mapping, score), (best_ion, best_mode))
    if isinstance(win[0], (list, tuple)) and len(win[0]) >= 3:
        return win[0][2]
    # flat format: (tweezed_ion, mapping, score, ...)
    if len(win) >= 3:
        return win[2]
    return None

def _extract_tweezed_ion(win):
    """Return tweezed ion index (or None) from a winner entry."""
    if not isinstance(win, (list, tuple)):
        return None
    if isinstance(win[0], (list, tuple)) and len(win[0]) >= 1:
        return win[0][0]
    return win[0]


In [11]:
P_opt = [4e-3]
w0 = 1e-6
f_rf_r = 1e6
f_rf_a = f_rf_r/3
results_test_2_7 = []
results_test_9 = []
results_test_11 = []
results_test_13 = []
results_test_15 = []

In [12]:
for Ni in range(2, 8):
    winners = run_optimal_mode_selection_tweezed_only_test(
        omega_tweezer,
        linewidths,
        omega_res,
        m,
        mode_calc_r,
        Ni,
        f_rf_r,
        f_rf_a,
        P_opt,
        w0,
        max_tweezed=1,
    )
    results_test_2_7.append((Ni, winners))


for Ni in range(9, 10):
    winners = run_optimal_mode_selection_tweezed_only_test(
        omega_tweezer,
        linewidths,
        omega_res,
        m,
        mode_calc_r,
        Ni,
        f_rf_r,
        f_rf_a,
        P_opt,
        w0,
        max_tweezed=1,
    )
    results_test_9.append((Ni, winners))

for Ni in range(11, 12):
    winners = run_optimal_mode_selection_tweezed_only_test(
        omega_tweezer,
        linewidths,
        omega_res,
        m,
        mode_calc_r,
        Ni,
        f_rf_r,
        f_rf_a,
        P_opt,
        w0,
        max_tweezed=1,
    )
    results_test_11.append((Ni, winners))

In [16]:
results_test_11

[(11,
  [((0,
     (0, 9, 10, 2, 1, 3, 5, 4, 8, 7, 6),
     np.float64(-0.03460038986654124),
     [0.004]),
    (5, 3)),
   ((0,
     (0, 9, 10, 2, 1, 3, 5, 6, 8, 7, 4),
     np.float64(-0.03460038986654124),
     [0.004]),
    (5, 3)),
   ((0,
     (0, 9, 10, 2, 1, 3, 5, 8, 7, 4, 6),
     np.float64(-0.03460038986654124),
     [0.004]),
    (5, 3)),
   ((0,
     (0, 9, 10, 2, 1, 3, 5, 8, 7, 6, 4),
     np.float64(-0.03460038986654124),
     [0.004]),
    (5, 3)),
   ((0,
     (0, 9, 10, 2, 1, 3, 8, 4, 5, 7, 6),
     np.float64(-0.03460038986654124),
     [0.004]),
    (5, 3)),
   ((0,
     (0, 9, 10, 2, 1, 3, 8, 4, 7, 6, 5),
     np.float64(-0.03460038986654124),
     [0.004]),
    (5, 3)),
   ((0,
     (0, 9, 10, 2, 1, 3, 8, 6, 5, 7, 4),
     np.float64(-0.03460038986654124),
     [0.004]),
    (5, 3)),
   ((0,
     (0, 9, 10, 2, 1, 3, 8, 6, 7, 4, 5),
     np.float64(-0.03460038986654124),
     [0.004]),
    (5, 3)),
   ((0,
     (1, 0, 10, 2, 9, 3, 5, 4, 8, 7, 6),
     np.float64(-

In [20]:
results_test_9

[(9,
  [((0, (0, 7, 8, 6, 1, 5, 2, 3, 4), np.float64(0.03949529499451976), [0.004]),
    (5, 5)),
   ((0, (0, 7, 8, 6, 1, 5, 4, 2, 3), np.float64(0.03949529499451976), [0.004]),
    (5, 5))])]

In [21]:
results_test_2_7

[(2, [((0, (0, 1), np.float64(0.09246723324859095), [0.004]), (0, 0))]),
 (3, [((0, (0, 2, 1), np.float64(0.08231057621929504), [0.004]), (2, 1))]),
 (4, [((1, (1, 3, 0, 2), np.float64(-0.06746571172111229), [0.004]), (0, 1))]),
 (5,
  [((1, (1, 4, 0, 3, 2), np.float64(-0.059800980625206), [0.004]), (3, 3))]),
 (6,
  [((2, (2, 5, 0, 4, 1, 3), np.float64(0.05095359033822568), [0.004]),
    (0, 2))]),
 (7,
  [((0, (0, 5, 6, 1, 3, 2, 4), np.float64(0.045980959386874805), [0.004]),
    (1, 5)),
   ((0, (0, 5, 6, 1, 3, 4, 2), np.float64(0.045980959386874805), [0.004]),
    (1, 5))])]

In [15]:
for Ni in range(13, 14):
    winners = run_optimal_mode_selection_tweezed_only_test(
        omega_tweezer,
        linewidths,
        omega_res,
        m,
        mode_calc_r,
        Ni,
        f_rf_r,
        f_rf_a,
        P_opt,
        w0,
        max_tweezed=1,
    )
    results_test_13.append((Ni, winners))

KeyboardInterrupt: 

In [ ]:
for Ni in range(15, 16):
    winners = run_optimal_mode_selection_tweezed_only_test(
        omega_tweezer,
        linewidths,
        omega_res,
        m,
        mode_calc_r,
        Ni,
        f_rf_r,
        f_rf_a,
        P_opt,
        w0,
        max_tweezed=1,
    )
    results_test_13.append((Ni, winners))

In [ ]:
results_test = results_test_2_7 + results_test_9 + results_test_11 + results_test_13 + results_test_15

In [ ]:
untweezed_results = []
for Ni in range[2,3,4,5,6,7,9,11,13,15]:
    winners = run_optimal_mode_selection_untweezed_only_test(
        omega_tweezer,
        linewidths,
        omega_res,
        m,
        mode_calc_r,
        Ni,
        f_rf_r,
        f_rf_a,
        P_opt,
        w0,
    )
    untweezed_results.append((Ni, winners))

In [ ]:

# Tweezed winners only
x_tweezed = []
y_tweezed = []

for N_val, winners in results_test:
    if not winners:
        continue
    for win in winners:
        tweezed_ion = _extract_tweezed_ion(win)
        if tweezed_ion is None:
            continue   # skip untweezed
        score = _extract_score_from_win(win)
        if score is None:
            continue
        x_tweezed.append(N_val)
        y_tweezed.append(abs(score))

# Untweezed winners
x_untweezed = []
y_untweezed = []

for N_val, winners in untweezed_results:
    if not winners:
        continue
    for win in winners:
        score = _extract_score_from_win(win)
        if score is None:
            continue
        x_untweezed.append(N_val)
        y_untweezed.append(abs(score))

# Convert to arrays (safe if empty)
x_tweezed = np.asarray(x_tweezed, dtype=float)
y_tweezed = np.asarray(y_tweezed, dtype=float)
x_untweezed = np.asarray(x_untweezed, dtype=float)
y_untweezed = np.asarray(y_untweezed, dtype=float)

# Calculate the ratio (1/tweezed) / (1/untweezed) = untweezed / tweezed per N
Ns = np.unique(np.concatenate((x_tweezed, x_untweezed)))
ratios = []

for N in Ns:
    tweezed_scores = y_tweezed[x_tweezed == N]
    untweezed_scores = y_untweezed[x_untweezed == N]
    if len(tweezed_scores) == 0 or len(untweezed_scores) == 0:  # Avoid division by zero
        continue
    # Compute the ratio for each untweezed score to each tweezed score
    for tweezed_score in tweezed_scores:
        for untweezed_score in untweezed_scores:
            ratios.append((N, untweezed_score / tweezed_score))

# Separate the ratios into x and y for plotting
x_ratios = np.array([r[0] for r in ratios])
y_ratios = np.array([r[1] for r in ratios])

# Plot the ratios
plt.figure(figsize=(10, 6))
plt.scatter(x_ratios, y_ratios, color='blue', label='(1/Tweezed) / (1/Untweezed)')
plt.xlabel('N (Number of Ions)')
plt.ylabel('Ratio: Untweezed / Tweezed')
plt.title('Ratio of Untweezed to Tweezed Scores per N')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(x_tweezed, 1.0 / y_tweezed, color='green', label='Tweezed')
plt.scatter(x_untweezed, 1.0 / y_untweezed, color='gray', label='Untweezed', alpha=0.7)
plt.xlabel("N",fontsize = 12)
plt.ylabel(r"$\tau$",fontsize = 12)
plt.title("Parallel Optimization",fontsize = 14)
plt.grid(True)
plt.legend()
plt.show()